# 05 — SHAP Interpretability

Compute SHAP values for XGBoost and logistic regression models.

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
from sklearn.model_selection import train_test_split
from src.config import *
from src.data_extraction import load_all_data
from src.preprocessing import preprocess_expression_matrix, prepare_ml_arrays
from src.models import get_model_definitions, get_endosomal_features
from src.interpretability import compute_shap_values, shap_feature_importance, save_shap_values
from src.figures import set_publication_style, plot_shap_beeswarm, plot_shap_bar
set_publication_style()

In [ ]:
data = load_all_data()
matrix = preprocess_expression_matrix(data['mouse_matrix'])
X, y, names = prepare_ml_arrays(matrix)
endo = get_endosomal_features(names)
endo_idx = [names.index(f) for f in endo if f in names]
X_endo = X[:, endo_idx]
print(f'Endosomal features: {len(endo)}')
print(endo[:10])

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X_endo, y, test_size=0.3, random_state=42, stratify=y)
print(f'Train: {X_tr.shape}, Test: {X_te.shape}')

In [ ]:
# Fit XGBoost
models = get_model_definitions()
xgb_pipe = models['Endosomal-XGB']
xgb_pipe.fit(X_tr, y_tr)
print('XGBoost trained')

In [ ]:
# SHAP values
shap_vals, expected_val = compute_shap_values(xgb_pipe, X_tr, X_te, endo)
print(f'SHAP values shape: {shap_vals.shape}')
importance = shap_feature_importance(shap_vals, endo)
print('\nTop 10 features:')
print(importance.head(10).to_string(index=False))

In [ ]:
# Save
save_shap_values(shap_vals, endo, 'Endosomal-XGB')
importance.to_csv(RESULTS_DIR / 'shap_importance_xgb.csv', index=False)

In [ ]:
# Bar chart
plot_shap_bar(importance)
print('SHAP bar chart saved')

In [ ]:
# Beeswarm (requires shap installed)
try:
    plot_shap_beeswarm(shap_vals, X_te, endo)
    print('SHAP beeswarm saved')
except Exception as e:
    print(f'Beeswarm skipped: {e}')

In [ ]:
# Logistic regression SHAP
lr_pipe = models['Endosomal-Optimized']
lr_pipe.fit(X_tr, y_tr)
shap_lr, ev_lr = compute_shap_values(lr_pipe, X_tr, X_te, endo, model_type='linear')
imp_lr = shap_feature_importance(shap_lr, endo)
print('\nTop 10 features (LR SHAP):')
print(imp_lr.head(10).to_string(index=False))